# Warranty Forecasting — Modular Test Notebook

Run models independently, exercise the full pipeline, and probe edge cases.

**Related docs:** [PROJECT_UI.md](PROJECT_UI.md)

### Sections
1. Setup & config
2. Helpers (modular — edit after testing)
3. Load data + production / cost sheet
4. Run models independently
5. Full train pipeline (forecast + CPV + claim ratio + CM)
6. Annotated walk-forward
7. PPT export
8. Edge-case tests
9. Scratch / experiments

## 1. Setup & config
Activate the project venv before selecting the kernel if needed.

In [ ]:
from __future__ import annotations

import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from forecasting.config import (
    DATA_FILES,
    FORECAST_HORIZON,
    LOOKBACK_WINDOW,
    N_CV_FOLDS,
    OUTPUT_DIR,
    RANDOM_SEED,
    WARRANTY_MONTHS,
)

np.random.seed(RANDOM_SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("cwd:", PROJECT_ROOT)
print("python:", sys.executable)
print(f"horizon={FORECAST_HORIZON} lookback={LOOKBACK_WINDOW} folds={N_CV_FOLDS} warranty={WARRANTY_MONTHS}")

## 2. Helpers (edit these to change behaviour)

In [ ]:
def make_flat_production(n_months: int, volume: float = 25_000.0, unit_cost: float = 12_000.0):
    """Simple constant production + cost sheet (replace with real values)."""
    prod = np.full(n_months, float(volume))
    cost = prod * float(unit_cost)
    return prod, cost


def make_ramping_production(n_months: int, start: float = 20_000.0, growth: float = 0.01, unit_cost: float = 12_000.0):
    """Growing production for sensitivity tests (not used by the app — notebook only)."""
    idx = np.arange(n_months, dtype=float)
    prod = start * ((1.0 + growth) ** idx)
    return prod, prod * unit_cost


def inject_spike(claims: np.ndarray, month_idx: int, factor: float = 5.0) -> np.ndarray:
    out = claims.astype(float).copy()
    if 0 <= month_idx < len(out):
        out[month_idx] *= factor
    return out


def summarize_result(res: dict) -> pd.DataFrame:
    econ = res.get("economics") or {}
    cms = res.get("cm_sim") or {}
    return pd.DataFrame([
        {
            "part": res.get("part"),
            "best_model": res.get("best_model"),
            "has_cm": res.get("has_cm"),
            "cm_peak_fcok": cms.get("peak_fcok"),
            "cm_improvement_pct": cms.get("improvement_pct"),
            "avg_cpv": econ.get("avg_cpv"),
            "avg_claim_ratio": econ.get("avg_claim_ratio"),
            "forecast_sum_12m": float(np.sum(res.get("best_forecast", []))),
        }
    ])


print("helpers ready")

## 3. Load data + production / cost sheet

In [ ]:
from forecasting.data.loader import load_and_prepare, validate_claims_dataframe, build_monthly_series
from forecasting.economics import (
    month_input_template,
    parse_month_inputs,
    production_sheet_to_csv,
    load_production_sheet_csv,
    peak_fcok_month,
    list_fcok_months,
)

DATA_PATH = DATA_FILES[0] if DATA_FILES and Path(DATA_FILES[0]).exists() else str(PROJECT_ROOT / "data" / "Test_All.csv")
PART = None  # set after load

raw = load_and_prepare([DATA_PATH])
ok, msgs = validate_claims_dataframe(raw)
print("valid:", ok, "|", " · ".join(msgs))

parts = sorted(raw["Part Name"].dropna().unique().tolist())
PART = parts[0]
print("parts:", len(parts), "| using:", PART)

tmpl = month_input_template(raw, PART)
prod, cost = make_flat_production(len(tmpl))
tmpl["Production"] = prod
tmpl["Production_Cost"] = cost
production, costs, months = parse_month_inputs(tmpl)
print("months:", len(months), "| production[0]=", production[0], "| cost[0]=", costs[0])
print("peak FCO/K:", peak_fcok_month(raw, PART))
tmpl.head(3)

In [ ]:
# Optional: download-style CSV round-trip (same path as the Gradio UI)
csv_path = production_sheet_to_csv(tmpl, str(PROJECT_ROOT / "outputs" / f"prod_sheet_{PART.replace(' ', '_')}.csv"))
reloaded = load_production_sheet_csv(csv_path)
print("CSV written →", csv_path)
reloaded.head(2)

## 4. Run models independently
Each cell trains one model family on a small window dataset for a smoke test.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from forecasting.pipeline.runner import build_window_dataset
from forecasting.models.ml_models import fit_sarima

monthly = build_monthly_series(raw, PART, production=production, require_production=True)
claims = monthly["claim_count"].values.astype(float)
print("monthly shape:", monthly.shape, "| claims sum:", claims.sum())

# --- SARIMA (baseline) ---
sarima_fc = fit_sarima(claims, horizon=FORECAST_HORIZON)
print("SARIMA 12m forecast:", np.round(sarima_fc, 1))

In [ ]:
from forecasting.models.cnn_lstm import CnnLstmForecaster

W = min(LOOKBACK_WINDOW, max(4, len(claims) // 3))
scaler = MinMaxScaler()
c_sc = scaler.fit_transform(claims.reshape(-1, 1)).ravel()
exog = monthly[["production"]].values.astype(float)
e_sc = MinMaxScaler().fit_transform(exog)

X, y = build_window_dataset(c_sc, e_sc, W, 1)
print("CNN-LSTM windows:", None if X is None else X.shape)

if X is not None and len(X) >= 4:
    cnn = CnnLstmForecaster(
        lookback=W, n_features=X.shape[2], horizon=1,
        lr=1e-2, epochs=15, dropout=0.2, early_patience=3,
    )
    cnn.fit(X, y)
    pred_sc = float(cnn.predict(X[-1:])[0, 0])
    pred = float(scaler.inverse_transform([[pred_sc]])[0, 0])
    print("CNN-LSTM next-step pred (orig units):", round(max(0, pred), 2))
else:
    print("Not enough history for CNN-LSTM window dataset")

In [ ]:
from forecasting.models.nbeats import NBeatsForecaster
from forecasting.models.transformer import TransformerForecaster

if X is not None and len(X) >= 4:
    nbeats = NBeatsForecaster(
        lookback=W, n_features=X.shape[2], horizon=1,
        lr=5e-3, epochs=10, dropout=0.2, early_patience=3,
    )
    nbeats.fit(X, y)
    p_nb = float(scaler.inverse_transform([[float(nbeats.predict(X[-1:])[0, 0])]])[0, 0])
    print("N-BEATS next-step:", round(max(0, p_nb), 2))

    tr = TransformerForecaster(
        lookback=W, n_features=X.shape[2], horizon=1,
        epochs=10, dropout=0.2, early_patience=3,
    )
    tr.fit(X, y)
    p_tr = float(scaler.inverse_transform([[float(tr.predict(X[-1:])[0, 0])]])[0, 0])
    print("Transformer next-step:", round(max(0, p_tr), 2))
else:
    print("Skip N-BEATS / Transformer — insufficient windows")

## 5. Full train pipeline (forecast + CPV + claim ratio + CM)

In [ ]:
from forecasting.pipeline.runner import train_uploaded_part
from forecasting.dashboard.builder import make_forecast_df

# Fast path: SARIMA only. Change selected_models to include DL models as needed.
SELECTED = ["SARIMA"]  # e.g. ["CNN-LSTM"] or ["SARIMA", "Transformer"]

result = train_uploaded_part(
    raw,
    PART,
    retune=False,
    use_locked_params=True,
    selected_models=SELECTED,
    production=production,
    production_cost=costs,
    cm_enabled=True,
    cm_month=None,  # peak FCO/K chosen automatically when enabled
    cm_reduction_pct=20.0,
)

display(summarize_result(result))
display(make_forecast_df(result).head(6))

econ_hist = pd.DataFrame({
    "Month": [str(p) for p in result["monthly"]["period"]],
    "Claims": result["claim_vals"],
    "Production": result["production"],
    "CPV": result.get("cpv"),
    "Claim_Ratio": result.get("claim_ratio"),
})
econ_hist.tail(5)

## 6. Annotated walk-forward (Model Testing)

In [ ]:
from forecasting.pipeline.annotated_forecast import forecast_last_n_months_annotated, HAS_TF

print("TensorFlow / Keras-CNN-LSTM available:", HAS_TF)

ann = forecast_last_n_months_annotated(
    raw,
    PART,
    models=["SARIMA"],  # add "CNN-LSTM", "N-BEATS", "Transformer" as needed
    time_step=4,
    epochs=20,
    n_test_months=N_CV_FOLDS,
    countermeasure_start=None,
)
ann

## 7. PPT export

In [ ]:
try:
    from forecasting.dashboard.ppt_export import build_forecast_pptx

    ppt_path = build_forecast_pptx(
        [result],
        annotated_df=ann,
        output_path=str(PROJECT_ROOT / "outputs" / f"notebook_briefing_{PART.replace(' ', '_')}.pptx"),
    )
    print("PPTX →", ppt_path, "|", os.path.getsize(ppt_path), "bytes")
except ModuleNotFoundError as exc:
    print("Install python-pptx in this kernel:\n  python -m pip install python-pptx\n", exc)

## 8. Edge-case tests

In [ ]:
# 8a. Missing production → must raise
from forecasting.economics import parse_month_inputs

bad = tmpl.copy()
bad.loc[0, "Production"] = np.nan
try:
    parse_month_inputs(bad)
    print("FAIL: expected ValueError for missing production")
except ValueError as e:
    print("PASS missing production:", e)

In [ ]:
# 8b. Extreme spike in claims (mutates a copy of monthly via custom production train)
# Rebuild production sheet but spike historical claims by training on a filtered raw subset is heavy;
# instead spike the series used for SARIMA smoke test.

spiked = inject_spike(claims, month_idx=len(claims) // 2, factor=8.0)
fc_spike = fit_sarima(spiked, horizon=6)
print("baseline last claim:", claims[-1], "| spiked mid:", spiked[len(claims)//2])
print("SARIMA after spike (6m):", np.round(fc_spike, 1))

In [ ]:
# 8c. Countermeasure on / off comparison
res_off = train_uploaded_part(
    raw, PART, retune=False, selected_models=["SARIMA"],
    production=production, production_cost=costs,
    cm_enabled=False, cm_reduction_pct=20,
)
res_on = train_uploaded_part(
    raw, PART, retune=False, selected_models=["SARIMA"],
    production=production, production_cost=costs,
    cm_enabled=True, cm_reduction_pct=40,
)
cmp = pd.DataFrame({
    "month": [str(p) for p in res_off["future_periods"]],
    "no_cm": np.round(res_off["best_forecast"], 1),
    "cm_40pct": np.round(res_on["best_forecast"], 1),
})
cmp["delta"] = cmp["cm_40pct"] - cmp["no_cm"]
print("CM peak FCO/K:", (res_on.get("cm_sim") or {}).get("peak_fcok"))
print("improvement %:", (res_on.get("cm_sim") or {}).get("improvement_pct"))
cmp.head(6)

In [ ]:
# 8d. Zero / tiny production months rejected
tiny = tmpl.copy()
tiny.loc[1, "Production"] = 0
try:
    parse_month_inputs(tiny)
    print("FAIL: expected rejection of zero production")
except ValueError as e:
    print("PASS zero production:", e)

## 9. Scratch / experiments
Add cells below to tweak lookback, CM %, model lists, or cost assumptions.

In [ ]:
# Example: change part and re-run section 5
# PART = parts[1]
# tmpl = month_input_template(raw, PART)
# prod, cost = make_flat_production(len(tmpl), volume=30_000, unit_cost=10_000)
# ...
print("Ready for extension. Available parts (first 10):", parts[:10])